In [1]:
from datasets import load_dataset

dataset = load_dataset("qintongli/GSM-Plus")
print(dataset)
ds = load_dataset("Maxwell-Jia/AIME_2024")
print(ds)

/home/iml/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    test: Dataset({
        features: ['question', 'solution', 'answer', 'perturbation_type', 'seed_question', 'seed_solution', 'seed_answer'],
        num_rows: 10552
    })
    testmini: Dataset({
        features: ['question', 'solution', 'answer', 'perturbation_type', 'seed_question', 'seed_solution', 'seed_answer'],
        num_rows: 2400
    })
})
DatasetDict({
    train: Dataset({
        features: ['ID', 'Problem', 'Solution', 'Answer'],
        num_rows: 30
    })
})


In [2]:
%pip install transformers datasets peft accelerate scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
# data_loader.py

from datasets import load_dataset
import json
import re

def extract_answer_gsm(answer_text):
    # GSM format: ".... #### 42"
    match = re.search(r"####\s*(-?\d+)", answer_text)
    return match.group(1) if match else None


def load_gsm_dataset():
    dataset = load_dataset("gsm8k", "main", split="test")

    data = []
    for item in dataset:
        data.append({
            "question": item["question"],
            "answer": extract_answer_gsm(item["answer"])
        })
    return data


def load_aime_dataset():
    dataset = load_dataset("Maxwell-Jia/AIME_2024", split="train")

    data = []
    for item in dataset:
        data.append({
            "question": item["Problem"],
            "answer": str(item["Answer"])
        })
    return data

In [4]:
# model_utils.py

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_model(model_name="Qwen/Qwen3-0.6B"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

    return tokenizer, model


def generate_with_entropy(model, tokenizer, prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            return_dict_in_generate=True,
            output_scores=True
        )

    generated_tokens = outputs.sequences[0]
    scores = outputs.scores  # logits per step

    entropies = []

    for step_logits in scores:
        probs = F.softmax(step_logits[0], dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-9)).item()
        entropies.append(entropy)

    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return generated_text, entropies

/home/iml/miniconda3/envs/thesis/lib/python3.14/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [5]:
# prompts.py

def cot_prompt(question):
    return f"""Solve the following problem step by step:

Question: {question}
Answer:"""


def pot_prompt(question):
    return f"""Write a Python program to solve the problem:

Question: {question}
Program:"""

In [12]:
# experiment.py
import re
from tqdm import tqdm

def extract_final_number(text):
    match = re.findall(r"-?\d+", text)
    return match[-1] if match else None


def run_experiment(dataset, tokenizer, model):
    results = []

    for item in tqdm(dataset):
        prompt = cot_prompt(item["question"])

        output, entropies = generate_with_entropy(model, tokenizer, prompt)

        pred = extract_final_number(output)
        correct = (pred == item["answer"])

        results.append({
            "question": item["question"],
            "prediction": pred,
            "ground_truth": item["answer"],
            "correct": correct,
            "entropy_mean": sum(entropies)/len(entropies),
            "entropy_max": max(entropies),
            "entropy_sequence": entropies
        })

    return results

In [13]:
# analysis.py
from sklearn.metrics import roc_auc_score

def compute_auroc(results):
    y_true = [0 if r["correct"] else 1 for r in results]  # 1 = failure
    y_scores = [r["entropy_mean"] for r in results]

    return roc_auc_score(y_true, y_scores)

In [15]:
# Load model
tokenizer, model = load_model()

# Load datasets
gsm_data = load_gsm_dataset()
aime_data = load_aime_dataset()

# Subsample for speed (IMPORTANT)
gsm_data = gsm_data[:200]
aime_data = aime_data[:100]

# Run
gsm_results = run_experiment(gsm_data, tokenizer, model)

aime_results = run_experiment(aime_data, tokenizer, model)

# Evaluate
print("GSM AUROC:", compute_auroc(gsm_results))
print("AIME AUROC:", compute_auroc(aime_results))

100%|██████████| 30/30 [26:46<00:00, 53.55s/it]

GSM AUROC: 0.5089376915219612
AIME AUROC: nan



/home/iml/miniconda3/envs/thesis/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
